# Simulación del Mundial 2026

En este notebook se simulará el Mundial 2026 utilizando el modelo predictivo entrenado previamente.

La simulación tendrá en cuenta:

- Ranking FIFA
- Puntos FIFA
- Rating Elo
- Forma reciente (últimos 5 partidos)

Además, se incorporará el RMSE del modelo como fuente de incertidumbre para reflejar la variabilidad real de un partido de fútbol.

El objetivo final es estimar la probabilidad de que cada selección alcance cada fase del torneo.

In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

In [3]:
dataset_modelado = pd.read_csv(
    "../data/processed/dataset_modelado.csv"
)

In [4]:
selecciones = pd.read_csv(
    "../data/processed/selecciones_actuales_mundial.csv"
)

selecciones.head()

,team,rank,total_points,elo,date,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,Mexico,15.0,1675.75,1835.0,2025-11-18,4.0,3.0,7.0,-4.0
1,South Africa,61.0,1426.73,1531.0,2025-12-29,10.0,8.0,4.0,4.0
2,South Korea,22.0,1599.45,1784.0,2025-11-18,10.0,9.0,6.0,3.0
3,Czech Republic,44.0,1487.00,1731.0,2025-11-17,11.0,6.0,2.0,4.0
4,Canada,27.0,1559.15,1802.0,2025-11-18,6.0,3.0,2.0,1.0


In [5]:
selecciones.shape

(48, 9)

In [6]:
selecciones = selecciones.drop(
    columns=["date"]
)

selecciones.shape

(48, 8)

In [7]:
selecciones.head()

,team,rank,total_points,elo,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,Mexico,15.0,1675.75,1835.0,4.0,3.0,7.0,-4.0
1,South Africa,61.0,1426.73,1531.0,10.0,8.0,4.0,4.0
2,South Korea,22.0,1599.45,1784.0,10.0,9.0,6.0,3.0
3,Czech Republic,44.0,1487.00,1731.0,11.0,6.0,2.0,4.0
4,Canada,27.0,1559.15,1802.0,6.0,3.0,2.0,1.0


In [8]:
selecciones.columns.tolist()

['team',
 'rank',
 'total_points',
 'elo',
 'last5_points',
 'last5_goals_for',
 'last5_goals_against',
 'last5_goal_balance']

In [9]:
modelo = joblib.load(
    "../models/modelo_regresion_lineal.pkl"
)
scaler = joblib.load(
    "../models/scaler.pkl"
)

## Función para obtener los datos de una selección

Para simular partidos necesitaremos acceder rápidamente a las características de cada selección.

Esta función devolverá todas las variables de una selección concreta.

In [10]:
equipos_dict = {}

for _, fila in selecciones.iterrows():

    equipos_dict[
        fila["team"]
    ] = fila


def obtener_equipo(nombre):

    return equipos_dict[nombre]

In [11]:
obtener_equipo("Spain")

team                     Spain
rank                       1.0
total_points           1877.18
elo                     2171.0
last5_points              13.0
last5_goals_for           17.0
last5_goals_against        9.0
last5_goal_balance         8.0
Name: 28, dtype: object

## Simulación de un partido

Esta función utilizará el modelo entrenado para predecir la diferencia de goles entre dos selecciones.

Posteriormente se añadirá un error aleatorio basado en el RMSE del modelo para reflejar la incertidumbre real de un partido de fútbol.

Finalmente la diferencia de goles se transformará en un resultado concreto.

## Construcción de las variables de un partido

Para cada enfrentamiento se calcularán las diferencias entre ambas selecciones utilizando las mismas variables empleadas durante el entrenamiento del modelo.

De esta forma la simulación utilizará exactamente la misma información que el modelo vio durante su entrenamiento.

In [12]:
def crear_partido(equipo_local, equipo_visitante):

    local = obtener_equipo(equipo_local)
    visitante = obtener_equipo(equipo_visitante)

    return np.array([[
    1,
    10,
    local["rank"] - visitante["rank"],
    local["total_points"] - visitante["total_points"],
    local["elo"] - visitante["elo"],
    local["last5_points"] - visitante["last5_points"],
    local["last5_goals_for"] - visitante["last5_goals_for"],
    local["last5_goals_against"] - visitante["last5_goals_against"],
    local["last5_goal_balance"] - visitante["last5_goal_balance"]
]])

In [13]:
crear_partido(
    "Spain",
    "Uruguay"
)

array([[  1.  ,  10.  , -15.  , 204.56, 281.  ,   7.  ,  15.  ,   6.  ,
          9.  ]])

## Predicción de la diferencia de goles

Utilizamos el modelo entrenado para predecir la diferencia de goles esperada entre dos selecciones.

Posteriormente se añadirá una perturbación aleatoria basada en el RMSE para reflejar la incertidumbre de un partido real.

In [14]:
rmse = 1.6435
rmse_simulacion = 0.8

def simular_partido(equipo_local, equipo_visitante):

    partido = crear_partido(
        equipo_local,
        equipo_visitante
    )

    partido_scaled = scaler.transform(
        partido
    )

    prediccion = modelo.predict(
        partido_scaled
    )[0]

    error = np.random.normal(
        loc=0,
        scale=rmse_simulacion
    )

    diferencia_final = prediccion + error

    return diferencia_final

In [15]:
for _ in range(10):

    print(
        simular_partido(
            "Spain",
            "Uruguay"
        )
    )

0.9985879325252668
-0.03174697505810198
0.34905140950590985
1.1236724105689848
0.5492678920331762
0.41821075755345094
-0.2213558539978202
0.11273291211380287
1.5798860000518773
0.3602370774726893


In [16]:
dataset_modelado["goal_diff"].describe()

count    16704.000000
mean         0.487787
std          2.100759
min        -14.000000
25%         -1.000000
50%          0.000000
75%          2.000000
max         17.000000
Name: goal_diff, dtype: float64

## Conversión de diferencia de goles en marcador

El modelo predice una diferencia de goles esperada.

Para obtener resultados más realistas se transforma dicha diferencia en un marcador concreto, permitiendo calcular puntos, goles a favor, goles en contra y diferencia de goles para la clasificación del torneo.

In [17]:
def generar_marcador(diferencia):

    diferencia_redondeada = int(round(diferencia))

    if diferencia_redondeada == 0:

        goles = np.random.randint(0, 4)

        return goles, goles

    goles_base = np.random.randint(0, 3)

    if diferencia_redondeada > 0:

        return (
            goles_base + diferencia_redondeada,
            goles_base
        )

    else:

        return (
            goles_base,
            goles_base + abs(diferencia_redondeada)
        )

In [18]:
for _ in range(10):

    diferencia = simular_partido(
        "Spain",
        "Uruguay"
    )

    print(
        diferencia,
        generar_marcador(diferencia)
    )

3.27741839852657 (3, 0)
0.5974901681178617 (2, 1)
1.7668156341728816 (4, 2)
0.9645278680658261 (3, 2)
1.3883818267063692 (1, 0)
0.012611621943529938 (2, 2)
-0.6858870830293917 (1, 2)
2.763564757044294 (5, 2)
1.8805581752777603 (4, 2)
1.6667789332578071 (3, 1)


## Simulación de un grupo

En esta sección simularemos todos los partidos de un grupo y construiremos la clasificación siguiendo las normas FIFA:

1. Puntos
2. Diferencia de goles
3. Goles a favor

In [19]:
grupo_a = [
    "Mexico",
    "South Africa",
    "South Korea",
    "Czech Republic"
]

In [20]:
tabla_grupo = pd.DataFrame({
    "team": grupo_a,
    "PJ": 0,
    "PG": 0,
    "PE": 0,
    "PP": 0,
    "GF": 0,
    "GC": 0,
    "DG": 0,
    "PTS": 0
})

tabla_grupo

,team,PJ,PG,PE,PP,GF,GC,DG,PTS
0,Mexico,0,0,0,0,0,0,0,0
1,South Africa,0,0,0,0,0,0,0,0
2,South Korea,0,0,0,0,0,0,0,0
3,Czech Republic,0,0,0,0,0,0,0,0


In [21]:
partidos_grupo = []

for i in range(len(grupo_a)):

    for j in range(i + 1, len(grupo_a)):

        partidos_grupo.append(
            (
                grupo_a[i],
                grupo_a[j]
            )
        )

partidos_grupo

[('Mexico', 'South Africa'),
 ('Mexico', 'South Korea'),
 ('Mexico', 'Czech Republic'),
 ('South Africa', 'South Korea'),
 ('South Africa', 'Czech Republic'),
 ('South Korea', 'Czech Republic')]

## Actualización de la clasificación

Después de cada partido se actualizarán:

- Partidos jugados
- Victorias
- Empates
- Derrotas
- Goles a favor
- Goles en contra
- Diferencia de goles
- Puntos

In [22]:
def actualizar_tabla(
    tabla,
    local,
    visitante,
    goles_local,
    goles_visitante
):

    # Partidos jugados

    tabla.loc[local, "PJ"] += 1
    tabla.loc[visitante, "PJ"] += 1

    # Goles

    tabla.loc[local, "GF"] += goles_local
    tabla.loc[local, "GC"] += goles_visitante

    tabla.loc[visitante, "GF"] += goles_visitante
    tabla.loc[visitante, "GC"] += goles_local

    # Resultado

    if goles_local > goles_visitante:

        tabla.loc[local, "PG"] += 1
        tabla.loc[local, "PTS"] += 3

        tabla.loc[visitante, "PP"] += 1

    elif goles_local < goles_visitante:

        tabla.loc[visitante, "PG"] += 1
        tabla.loc[visitante, "PTS"] += 3

        tabla.loc[local, "PP"] += 1

    else:

        tabla.loc[local, "PE"] += 1
        tabla.loc[visitante, "PE"] += 1

        tabla.loc[local, "PTS"] += 1
        tabla.loc[visitante, "PTS"] += 1

    return tabla

## Simulación completa del grupo

Se simulan los seis partidos del grupo.

Tras cada encuentro se actualiza la clasificación con los puntos y estadísticas correspondientes.

In [23]:
def simular_grupo(equipos):

    tabla = pd.DataFrame({
        "team": equipos,
        "PJ": 0,
        "PG": 0,
        "PE": 0,
        "PP": 0,
        "GF": 0,
        "GC": 0,
        "DG": 0,
        "PTS": 0
    })

    tabla = tabla.set_index(
        "team"
    )

    partidos = []

    resultados_partidos = []

    for i in range(len(equipos)):

        for j in range(i + 1, len(equipos)):

            partidos.append(
                (
                    equipos[i],
                    equipos[j]
                )
            )

    for local, visitante in partidos:

        diferencia = simular_partido(
            local,
            visitante
        )

        goles_local, goles_visitante = generar_marcador(
            diferencia
        )

        resultados_partidos.append(
            {
                "local": local,
                "visitante": visitante,
                "goles_local": goles_local,
                "goles_visitante": goles_visitante
            }
        )

        actualizar_tabla(
            tabla,
            local,
            visitante,
            goles_local,
            goles_visitante
        )

    tabla["DG"] = (
        tabla["GF"]
        - tabla["GC"]
    )

    tabla = tabla.sort_values(
        by=["PTS", "DG", "GF"],
        ascending=False
    )

    tabla = tabla.reset_index()

    tabla["posicion"] = range(
        1,
        len(tabla) + 1
    )

    tabla = tabla[
        [
            "posicion",
            "team",
            "PJ",
            "PG",
            "PE",
            "PP",
            "GF",
            "GC",
            "DG",
            "PTS"
        ]
    ]

    partidos_df = pd.DataFrame(
        resultados_partidos
    )

    return tabla, partidos_df

In [24]:
clasificacion, partidos = simular_grupo(
    grupo_a
)

In [25]:
partidos["resultado"] = (
    partidos["goles_local"].astype(str)
    + "-"
    + partidos["goles_visitante"].astype(str)
)


In [26]:
partidos

,local,visitante,goles_local,goles_visitante,resultado
0,Mexico,South Africa,4,1,4-1
1,Mexico,South Korea,2,1,2-1
2,Mexico,Czech Republic,2,1,2-1
3,South Africa,South Korea,3,2,3-2
4,South Africa,Czech Republic,0,1,0-1
5,South Korea,Czech Republic,2,2,2-2


In [27]:
clasificacion

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS
0,1,Mexico,3,3,0,0,8,3,5,9
1,2,Czech Republic,3,1,1,1,4,4,0,4
2,3,South Africa,3,1,0,2,4,7,-3,3
3,4,South Korea,3,0,1,2,5,7,-2,1


## Obtener clasificados de un grupo

Una vez simulado un grupo, necesitamos identificar:

- 1º clasificado
- 2º clasificado
- 3º clasificado
- 4º clasificado

Esta información será necesaria posteriormente para:

- Clasificados directos (1º y 2º)
- Ranking de mejores terceros
- Cruces de eliminación directa

In [28]:
def obtener_clasificados(clasificacion):

    primero = clasificacion.iloc[0]
    segundo = clasificacion.iloc[1]
    tercero = clasificacion.iloc[2]
    cuarto = clasificacion.iloc[3]

    return primero, segundo, tercero, cuarto

In [29]:
primero, segundo, tercero, cuarto = obtener_clasificados(
    clasificacion
)

In [30]:
print("1º:", primero["team"])
print("2º:", segundo["team"])
print("3º:", tercero["team"])
print("4º:", cuarto["team"])

1º: Mexico
2º: Czech Republic
3º: South Africa
4º: South Korea


## Definición de los grupos del Mundial

Vamos a almacenar los 12 grupos del Mundial en un diccionario.

Esto permitirá recorrer todos los grupos automáticamente y simular toda la fase de grupos mediante un único bucle.

In [31]:
grupos = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],

    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],

    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],

    "D": ["United States", "Paraguay", "Australia", "Turkey"],

    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],

    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],

    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],

    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],

    "I": ["France", "Senegal", "Iraq", "Norway"],

    "J": ["Argentina", "Algeria", "Austria", "Jordan"],

    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],

    "L": ["England", "Croatia", "Ghana", "Panama"]
}

## Simulación completa de la fase de grupos

Se simularán automáticamente los 12 grupos del Mundial.

Para cada grupo se almacenará:

- Clasificación final
- Resultados de los partidos
- Primer clasificado
- Segundo clasificado
- Tercer clasificado
- Cuarto clasificado

In [32]:
def simular_fase_grupos(grupos):

    clasificaciones = {}

    partidos_grupos = {}

    primeros = []

    segundos = []

    terceros = []

    cuartos = []

    for nombre_grupo, equipos in grupos.items():

        clasificacion, partidos = simular_grupo(
            equipos
        )

        clasificaciones[nombre_grupo] = clasificacion

        partidos_grupos[nombre_grupo] = partidos

        primero = clasificacion.iloc[0].copy()
        primero["grupo"] = nombre_grupo
        primeros.append(primero)

        segundo = clasificacion.iloc[1].copy()
        segundo["grupo"] = nombre_grupo
        segundos.append(segundo)

        tercero = clasificacion.iloc[2].copy()
        tercero["grupo"] = nombre_grupo
        terceros.append(tercero)

        cuarto = clasificacion.iloc[3].copy()
        cuarto["grupo"] = nombre_grupo
        cuartos.append(cuarto)

    primeros = pd.DataFrame(primeros)

    segundos = pd.DataFrame(segundos)

    terceros = pd.DataFrame(terceros)

    cuartos = pd.DataFrame(cuartos)

    return (
        clasificaciones,
        partidos_grupos,
        primeros,
        segundos,
        terceros,
        cuartos
    )

In [33]:
(
    clasificaciones,
    partidos_grupos,
    primeros,
    segundos,
    terceros,
    cuartos
) = simular_fase_grupos(
    grupos
)

In [34]:
clasificaciones["A"]

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS
0,1,Mexico,3,2,1,0,5,2,3,7
1,2,Czech Republic,3,2,1,0,5,3,2,7
2,3,South Korea,3,0,1,2,5,7,-2,1
3,4,South Africa,3,0,1,2,2,5,-3,1


In [35]:
clasificaciones["H"]

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS
0,1,Spain,3,3,0,0,8,3,5,9
1,2,Uruguay,3,1,1,1,7,7,0,4
2,3,Saudi Arabia,3,0,2,1,5,6,-1,2
3,4,Cape Verde,3,0,1,2,2,6,-4,1


In [36]:
partidos_grupos["H"]

,local,visitante,goles_local,goles_visitante
0,Spain,Cape Verde,3,0
1,Spain,Saudi Arabia,3,2
2,Spain,Uruguay,2,1
3,Cape Verde,Saudi Arabia,0,0
4,Cape Verde,Uruguay,2,3
5,Saudi Arabia,Uruguay,3,3


In [37]:
primeros[["team","PTS","DG","GF"]]

,team,PTS,DG,GF
0,Mexico,7,3,5
0,Switzerland,7,2,5
0,Morocco,6,1,8
0,Turkey,7,2,7
0,Ecuador,7,5,11
0,Japan,9,4,6
0,Iran,5,1,8
0,Spain,9,5,8
0,France,7,3,9
0,Argentina,9,6,10


In [38]:
terceros[["team","PTS","DG","GF"]]

,team,PTS,DG,GF
2,South Korea,1,-2,5
2,Qatar,2,-1,3
2,Scotland,3,0,7
2,Australia,2,-1,4
2,Ivory Coast,2,-3,4
2,Tunisia,1,-3,3
2,Egypt,4,0,7
2,Saudi Arabia,2,-1,5
2,Senegal,1,-2,5
2,Jordan,3,-1,5


## Selección de los 8 mejores terceros

Los terceros de cada grupo se ordenan utilizando los criterios FIFA:

1. Puntos
2. Diferencia de goles
3. Goles a favor

Los 8 mejores avanzan a dieciseisavos de final.

In [39]:
ranking_terceros = terceros.sort_values(
    by=["PTS", "DG", "GF"],
    ascending=False
).reset_index(drop=True)

ranking_terceros

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS,grupo
0,3,Egypt,3,1,1,1,7,7,0,4,G
1,3,Croatia,3,1,1,1,4,4,0,4,L
2,3,Scotland,3,1,0,2,7,7,0,3,C
3,3,Jordan,3,1,0,2,5,6,-1,3,J
4,3,Saudi Arabia,3,0,2,1,5,6,-1,2,H
5,3,Australia,3,0,2,1,4,5,-1,2,D
6,3,Uzbekistan,3,0,2,1,4,5,-1,2,K
7,3,Qatar,3,0,2,1,3,4,-1,2,B
8,3,Ivory Coast,3,0,2,1,4,7,-3,2,E
9,3,South Korea,3,0,1,2,5,7,-2,1,A


In [40]:
ranking_primeros = primeros.sort_values(
    by=["PTS", "DG", "GF"],
    ascending=False
).reset_index(drop=True)

ranking_primeros

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS,grupo
0,1,Argentina,3,3,0,0,10,4,6,9,J
1,1,Spain,3,3,0,0,8,3,5,9,H
2,1,Japan,3,3,0,0,6,2,4,9,F
3,1,Ecuador,3,2,1,0,11,6,5,7,E
4,1,France,3,2,1,0,9,6,3,7,I
5,1,Mexico,3,2,1,0,5,2,3,7,A
6,1,Colombia,3,2,1,0,8,6,2,7,K
7,1,Turkey,3,2,1,0,7,5,2,7,D
8,1,Switzerland,3,2,1,0,5,3,2,7,B
9,1,Panama,3,2,1,0,5,3,2,7,L


In [41]:
mejores_terceros = ranking_terceros.head(8)

mejores_terceros[
    ["team", "PTS", "DG", "GF"]
]

,team,PTS,DG,GF
0,Egypt,4,0,7
1,Croatia,4,0,4
2,Scotland,3,0,7
3,Jordan,3,-1,5
4,Saudi Arabia,2,-1,5
5,Australia,2,-1,4
6,Uzbekistan,2,-1,4
7,Qatar,2,-1,3


In [42]:
terceros_eliminados = ranking_terceros.tail(4)

terceros_eliminados[
    ["team", "PTS", "DG", "GF"]
]

,team,PTS,DG,GF
8,Ivory Coast,2,-3,4
9,South Korea,1,-2,5
10,Senegal,1,-2,5
11,Tunisia,1,-3,3


In [43]:
primeros.head()

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS,grupo
0,1,Mexico,3,2,1,0,5,2,3,7,A
0,1,Switzerland,3,2,1,0,5,3,2,7,B
0,1,Morocco,3,2,0,1,8,7,1,6,C
0,1,Turkey,3,2,1,0,7,5,2,7,D
0,1,Ecuador,3,2,1,0,11,6,5,7,E


In [44]:
terceros.head()

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS,grupo
2,3,South Korea,3,0,1,2,5,7,-2,1,A
2,3,Qatar,3,0,2,1,3,4,-1,2,B
2,3,Scotland,3,1,0,2,7,7,0,3,C
2,3,Australia,3,0,2,1,4,5,-1,2,D
2,3,Ivory Coast,3,0,2,1,4,7,-3,2,E


## Preparación de los cruces de dieciseisavos

Ya conocemos los 32 equipos clasificados:

- 12 primeros de grupo
- 12 segundos de grupo
- 8 mejores terceros

Para construir los cruces oficiales del Mundial necesitamos poder acceder rápidamente a cualquier clasificado mediante:

- Grupo
- Posición obtenida

Por ejemplo:

- 1º Grupo A
- 2º Grupo H
- 3º Grupo E

Para ello se crea un diccionario que relaciona cada combinación (grupo, posición) con el nombre de la selección clasificada.

In [45]:
clasificados = pd.concat(
    [
        primeros.assign(tipo="1º"),
        segundos.assign(tipo="2º"),
        mejores_terceros.assign(tipo="3º")
    ],
    ignore_index=True
)

clasificados

,posicion,team,PJ,PG,PE,PP,GF,GC,DG,PTS,grupo,tipo
0,1,Mexico,3,2,1,0,5,2,3,7,A,1º
1,1,Switzerland,3,2,1,0,5,3,2,7,B,1º
2,1,Morocco,3,2,0,1,8,7,1,6,C,1º
3,1,Turkey,3,2,1,0,7,5,2,7,D,1º
4,1,Ecuador,3,2,1,0,11,6,5,7,E,1º
5,1,Japan,3,3,0,0,6,2,4,9,F,1º
6,1,Iran,3,1,2,0,8,7,1,5,G,1º
7,1,Spain,3,3,0,0,8,3,5,9,H,1º
8,1,France,3,2,1,0,9,6,3,7,I,1º
9,1,Argentina,3,3,0,0,10,4,6,9,J,1º


In [46]:
clasificados.shape

(32, 12)

In [47]:
clasificados[
    ["grupo", "tipo", "team"]
].sort_values(
    by=["grupo", "tipo"]
)

,grupo,tipo,team
0,A,1º,Mexico
12,A,2º,Czech Republic
1,B,1º,Switzerland
13,B,2º,Canada
31,B,3º,Qatar
2,C,1º,Morocco
14,C,2º,Brazil
26,C,3º,Scotland
3,D,1º,Turkey
15,D,2º,Paraguay


In [48]:
terceros_clasificados = (
    mejores_terceros["grupo"]
    .sort_values()
    .tolist()
)

terceros_clasificados

['B', 'C', 'D', 'G', 'H', 'J', 'K', 'L']

In [49]:
equipos_por_grupo = {}

for _, fila in clasificados.iterrows():

    grupo = fila["grupo"]
    tipo = fila["tipo"]

    equipos_por_grupo[(grupo, tipo)] = fila["team"]

In [50]:
equipos_por_grupo[("A", "1º")]

'Mexico'

In [51]:
equipos_por_grupo[("A", "2º")]

'Czech Republic'

## Función auxiliar para obtener clasificados

Esta función simplifica la lectura de los cruces.

En lugar de escribir:

equipos_por_grupo[("A","1º")]

podremos escribir:

obtener_equipo("A","1º")

haciendo el código de los cruces mucho más legible.

In [52]:
def obtener_clasificado(grupo, posicion):

    return equipos_por_grupo[
        (grupo, posicion)
    ]

In [53]:
obtener_clasificado(
    "A",
    "1º"
)

'Mexico'

QUEDA VER BIEN LO DE LOS MEJROES 3 PORQUE PUEDE GENERAR DUDAS, EN FUNCION DE QUIENES PASEN 

## Construcción de los cruces oficiales

Se crea la estructura de los partidos de dieciseisavos.

Los cruces fijos entre primeros y segundos de grupo pueden generarse directamente.

Los cruces que dependen de terceros clasificados se completarán posteriormente mediante una función específica.

## Completar los 16 partidos de dieciseisavos

Los cruces que dependen de los mejores terceros se asignarán temporalmente utilizando el ranking de terceros.

Posteriormente sustituiremos esta lógica por la tabla oficial FIFA.

In [54]:
terceros_ordenados = (
    mejores_terceros["team"]
    .tolist()
)

terceros_ordenados

['Egypt',
 'Croatia',
 'Scotland',
 'Jordan',
 'Saudi Arabia',
 'Australia',
 'Uzbekistan',
 'Qatar']

In [55]:
cruces_terceros = {

    74: (
        equipos_por_grupo[("E", "1º")],
        terceros_ordenados[0]
    ),

    77: (
        equipos_por_grupo[("I", "1º")],
        terceros_ordenados[1]
    ),

    79: (
        equipos_por_grupo[("A", "1º")],
        terceros_ordenados[2]
    ),

    80: (
        equipos_por_grupo[("L", "1º")],
        terceros_ordenados[3]
    ),

    81: (
        equipos_por_grupo[("D", "1º")],
        terceros_ordenados[4]
    ),

    82: (
        equipos_por_grupo[("G", "1º")],
        terceros_ordenados[5]
    ),

    85: (
        equipos_por_grupo[("B", "1º")],
        terceros_ordenados[6]
    ),

    87: (
        equipos_por_grupo[("K", "1º")],
        terceros_ordenados[7]
    )
}

In [56]:
cruces_fijos = {

    73: ("A", "2º", "B", "2º"),

    75: ("F", "1º", "C", "2º"),

    76: ("C", "1º", "F", "2º"),

    78: ("E", "2º", "I", "2º"),

    83: ("K", "2º", "L", "2º"),

    84: ("H", "1º", "J", "2º"),

    86: ("J", "1º", "H", "2º"),

    88: ("D", "2º", "G", "2º")
}

In [57]:
def generar_cruces_fijos():

    partidos = []

    for numero, cruce in cruces_fijos.items():

        grupo1, pos1, grupo2, pos2 = cruce

        partidos.append({

            "partido": numero,

            "local": obtener_clasificado(
                grupo1,
                pos1
            ),

            "visitante": obtener_clasificado(
                grupo2,
                pos2
            )
        })

    return pd.DataFrame(
        partidos
    ).sort_values(
        "partido"
    )

In [58]:
cruces_16avos = generar_cruces_fijos()

cruces_16avos

,partido,local,visitante
0,73,Czech Republic,Canada
1,75,Japan,Brazil
2,76,Morocco,Netherlands
3,78,Germany,Norway
4,83,Portugal,England
5,84,Spain,Austria
6,86,Argentina,Uruguay
7,88,Paraguay,Belgium


## Estructura oficial de los cruces

Cada partido de eliminación directa se identifica mediante el número oficial FIFA.

Esto permitirá construir automáticamente:

- Dieciseisavos
- Octavos
- Cuartos
- Semifinales
- Final

siguiendo exactamente el cuadro oficial del Mundial 2026.

In [59]:
partidos_eliminatoria = {}

In [60]:
for _, fila in cruces_16avos.iterrows():

    partidos_eliminatoria[
        fila["partido"]
    ] = {
        "local": fila["local"],
        "visitante": fila["visitante"]
    }

partidos_eliminatoria

{73: {'local': 'Czech Republic', 'visitante': 'Canada'},
 75: {'local': 'Japan', 'visitante': 'Brazil'},
 76: {'local': 'Morocco', 'visitante': 'Netherlands'},
 78: {'local': 'Germany', 'visitante': 'Norway'},
 83: {'local': 'Portugal', 'visitante': 'England'},
 84: {'local': 'Spain', 'visitante': 'Austria'},
 86: {'local': 'Argentina', 'visitante': 'Uruguay'},
 88: {'local': 'Paraguay', 'visitante': 'Belgium'}}

In [61]:
for numero, (local, visitante) in cruces_terceros.items():

    partidos_eliminatoria[numero] = {

        "local": local,
        "visitante": visitante
    }

In [62]:
sorted(
    partidos_eliminatoria.keys()
)

[73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88]

In [63]:
partidos_eliminatoria[73]

{'local': 'Czech Republic', 'visitante': 'Canada'}

## Simulación de partidos eliminatorios

En las fases eliminatorias no puede haber empates.

Si el partido termina empatado tras los 90 minutos, se simulará una prórroga y, si fuera necesario, una tanda de penaltis.

Para simplificar el modelo, los empates se resolverán aleatoriamente entre ambos equipos.

In [64]:
def simular_eliminatoria(local, visitante):

    diferencia = simular_partido(
        local,
        visitante
    )

    goles_local, goles_visitante = generar_marcador(
        diferencia
    )

    if goles_local > goles_visitante:

        ganador = str(local)

    elif goles_visitante > goles_local:

        ganador = str(visitante)

    else:

        ganador = str (np.random.choice(
            [local, visitante])
        )

    return {
        "local": local,
        "visitante": visitante,
        "goles_local": goles_local,
        "goles_visitante": goles_visitante,
        "ganador": ganador
    }

In [65]:
for _ in range(5):

    print(
        simular_eliminatoria(
            "Spain",
            "Austria"
        )
    )

{'local': 'Spain', 'visitante': 'Austria', 'goles_local': 3, 'goles_visitante': 3, 'ganador': 'Spain'}
{'local': 'Spain', 'visitante': 'Austria', 'goles_local': 2, 'goles_visitante': 1, 'ganador': 'Spain'}
{'local': 'Spain', 'visitante': 'Austria', 'goles_local': 4, 'goles_visitante': 2, 'ganador': 'Spain'}
{'local': 'Spain', 'visitante': 'Austria', 'goles_local': 1, 'goles_visitante': 0, 'ganador': 'Spain'}
{'local': 'Spain', 'visitante': 'Austria', 'goles_local': 3, 'goles_visitante': 3, 'ganador': 'Spain'}


## Almacenamiento de resultados eliminatorios

Cada partido eliminatorio se almacenará utilizando su número oficial FIFA.

Esto permitirá construir automáticamente las siguientes rondas utilizando los ganadores de los partidos anteriores.

In [66]:
resultados_eliminatoria = {}

In [67]:
resultado_73 = simular_eliminatoria(
    partidos_eliminatoria[73]["local"],
    partidos_eliminatoria[73]["visitante"]
)

resultado_73

{'local': 'Czech Republic',
 'visitante': 'Canada',
 'goles_local': 4,
 'goles_visitante': 2,
 'ganador': 'Czech Republic'}

In [68]:
resultados_eliminatoria[73] = resultado_73

resultados_eliminatoria[73]

{'local': 'Czech Republic',
 'visitante': 'Canada',
 'goles_local': 4,
 'goles_visitante': 2,
 'ganador': 'Czech Republic'}

In [69]:
resultados_eliminatoria[73]["ganador"]

'Czech Republic'

## Simulación automática de las eliminatorias

Vamos a crear una función que juegue un partido eliminatorio a partir de su número oficial FIFA.

La función buscará los equipos participantes, simulará el partido y almacenará el ganador para que pueda utilizarse en rondas posteriores.

In [70]:
def jugar_partido_eliminatoria(numero_partido):

    local = partidos_eliminatoria[numero_partido]["local"]

    visitante = partidos_eliminatoria[numero_partido]["visitante"]

    resultado = simular_eliminatoria(
        local,
        visitante
    )

    resultados_eliminatoria[numero_partido] = resultado

    return resultado

In [71]:
jugar_partido_eliminatoria(73)

{'local': 'Czech Republic',
 'visitante': 'Canada',
 'goles_local': 0,
 'goles_visitante': 0,
 'ganador': 'Czech Republic'}

## Obtener ganadores de partidos anteriores

Las rondas siguientes utilizan ganadores de partidos previos.

Por ello creamos una función que devuelve automáticamente el ganador de cualquier partido ya disputado.

In [72]:
def ganador(numero_partido):

    return resultados_eliminatoria[
        numero_partido
    ]["ganador"]

In [73]:
ganador(73)

'Czech Republic'

## Construcción automática de los octavos de final

Según el cuadro oficial FIFA:

- Partido 89 = Ganador 74 vs Ganador 77
- Partido 90 = Ganador 73 vs Ganador 75
- Partido 91 = Ganador 76 vs Ganador 78
- Partido 92 = Ganador 79 vs Ganador 80
- Partido 93 = Ganador 83 vs Ganador 84
- Partido 94 = Ganador 81 vs Ganador 82
- Partido 95 = Ganador 86 vs Ganador 88
- Partido 96 = Ganador 85 vs Ganador 87

In [74]:
cruces_octavos = {

    89: (74, 77),
    90: (73, 75),
    91: (76, 78),
    92: (79, 80),

    93: (83, 84),
    94: (81, 82),
    95: (86, 88),
    96: (85, 87)
}

In [75]:
def jugar_octavo(numero_octavo):

    p1, p2 = cruces_octavos[numero_octavo]

    equipo1 = ganador(p1)

    equipo2 = ganador(p2)

    resultado = simular_eliminatoria(
        equipo1,
        equipo2
    )

    resultados_eliminatoria[numero_octavo] = resultado

    return resultado

In [76]:
for partido in range(73, 89):

    jugar_partido_eliminatoria(
        partido
    )

In [77]:
jugar_octavo(89)

{'local': 'Ecuador',
 'visitante': 'France',
 'goles_local': 0,
 'goles_visitante': 1,
 'ganador': 'France'}

## Estructura de cuartos de final

Según el cuadro oficial FIFA:

- 97 = Ganador 89 vs Ganador 90
- 98 = Ganador 93 vs Ganador 94
- 99 = Ganador 91 vs Ganador 92
- 100 = Ganador 95 vs Ganador 96

In [78]:
cruces_cuartos = {

    97: (89, 90),

    98: (93, 94),

    99: (91, 92),

    100: (95, 96)
}

## Función para simular cuartos de final

In [79]:
def jugar_cuarto(numero_cuarto):

    p1, p2 = cruces_cuartos[numero_cuarto]

    equipo1 = ganador(p1)

    equipo2 = ganador(p2)

    resultado = simular_eliminatoria(
        equipo1,
        equipo2
    )

    resultados_eliminatoria[numero_cuarto] = resultado

    return resultado

Semifinales

In [80]:
cruces_semifinales = {

    101: (97, 98),

    102: (99, 100)
}

In [81]:
def jugar_semifinal(numero_semifinal):

    p1, p2 = cruces_semifinales[numero_semifinal]

    equipo1 = ganador(p1)

    equipo2 = ganador(p2)

    resultado = simular_eliminatoria(
        equipo1,
        equipo2
    )

    resultados_eliminatoria[numero_semifinal] = resultado

    return resultado

FINAL

In [82]:
def jugar_final():

    equipo1 = ganador(101)

    equipo2 = ganador(102)

    resultado = simular_eliminatoria(
        equipo1,
        equipo2
    )

    resultados_eliminatoria[104] = resultado

    return resultado

## Simulación completa del cuadro eliminatorio

Esta función jugará automáticamente todas las rondas hasta obtener el campeón.

In [83]:
def simular_fase_final():

    resultados_eliminatoria.clear()

    for partido in range(73, 89):

        jugar_partido_eliminatoria(
            partido
        )

    for partido in range(89, 97):

        jugar_octavo(
            partido
        )

    for partido in range(97, 101):

        jugar_cuarto(
            partido
        )

    jugar_semifinal(101)

    jugar_semifinal(102)

    final = jugar_final()

    return final

In [84]:
simular_fase_final()

{'local': 'England',
 'visitante': 'Netherlands',
 'goles_local': 1,
 'goles_visitante': 1,
 'ganador': 'Netherlands'}

# Simulación completa del Mundial

Vamos a integrar todas las fases del torneo en una única función.

Cada ejecución generará:

- Fase de grupos
- Clasificados
- Mejores terceros
- Cruces eliminatorios
- Campeón

In [85]:
def simular_mundial():

    (
        clasificaciones,
        partidos_grupos,
        primeros,
        segundos,
        terceros,
        cuartos
    ) = simular_fase_grupos(
        grupos
    )

    ranking_terceros = (
        terceros
        .sort_values(
            by=["PTS", "DG", "GF"],
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    mejores_terceros = (
        ranking_terceros.head(8)
    )

    equipos_por_grupo = {}

    for _, fila in primeros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "1º")
        ] = fila["team"]

    for _, fila in segundos.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "2º")
        ] = fila["team"]

    for _, fila in mejores_terceros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "3º")
        ] = fila["team"]

    terceros_ordenados = (
        mejores_terceros["team"]
        .tolist()
    )

    cruces_terceros = {

        74: (
            equipos_por_grupo[("E", "1º")],
            terceros_ordenados[0]
        ),

        77: (
            equipos_por_grupo[("I", "1º")],
            terceros_ordenados[1]
        ),

        79: (
            equipos_por_grupo[("A", "1º")],
            terceros_ordenados[2]
        ),

        80: (
            equipos_por_grupo[("L", "1º")],
            terceros_ordenados[3]
        ),

        81: (
            equipos_por_grupo[("D", "1º")],
            terceros_ordenados[4]
        ),

        82: (
            equipos_por_grupo[("G", "1º")],
            terceros_ordenados[5]
        ),

        85: (
            equipos_por_grupo[("B", "1º")],
            terceros_ordenados[6]
        ),

        87: (
            equipos_por_grupo[("K", "1º")],
            terceros_ordenados[7]
        )
    }

    cruces_fijos = {

        73: (
            equipos_por_grupo[("A", "2º")],
            equipos_por_grupo[("B", "2º")]
        ),

        75: (
            equipos_por_grupo[("F", "1º")],
            equipos_por_grupo[("C", "2º")]
        ),

        76: (
            equipos_por_grupo[("C", "1º")],
            equipos_por_grupo[("F", "2º")]
        ),

        78: (
            equipos_por_grupo[("E", "2º")],
            equipos_por_grupo[("I", "2º")]
        ),

        83: (
            equipos_por_grupo[("K", "2º")],
            equipos_por_grupo[("L", "2º")]
        ),

        84: (
            equipos_por_grupo[("H", "1º")],
            equipos_por_grupo[("J", "2º")]
        ),

        86: (
            equipos_por_grupo[("J", "1º")],
            equipos_por_grupo[("H", "2º")]
        ),

        88: (
            equipos_por_grupo[("D", "2º")],
            equipos_por_grupo[("G", "2º")]
        )
    }

    partidos_eliminatoria = {}

    for numero, (local, visitante) in cruces_fijos.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    for numero, (local, visitante) in cruces_terceros.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    resultados_eliminatoria = {}

    def ganador(numero_partido):

        return resultados_eliminatoria[
            numero_partido
        ]["ganador"]

    def jugar(numero_partido):

        local = partidos_eliminatoria[
            numero_partido
        ]["local"]

        visitante = partidos_eliminatoria[
            numero_partido
        ]["visitante"]

        resultado = simular_eliminatoria(
            local,
            visitante
        )

        resultados_eliminatoria[
            numero_partido
        ] = resultado

    # Dieciseisavos

    for partido in range(73, 89):

        jugar(partido)

    # Octavos

    cruces_octavos = {

        89: (74, 77),
        90: (73, 75),
        91: (76, 78),
        92: (79, 80),

        93: (83, 84),
        94: (81, 82),
        95: (86, 88),
        96: (85, 87)
    }

    for partido, (p1, p2) in cruces_octavos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Cuartos

    cruces_cuartos = {

        97: (89, 90),

        98: (93, 94),

        99: (91, 92),

        100: (95, 96)
    }

    for partido, (p1, p2) in cruces_cuartos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Semifinales

    resultado = simular_eliminatoria(
        ganador(97),
        ganador(98)
    )

    resultados_eliminatoria[101] = resultado

    resultado = simular_eliminatoria(
        ganador(99),
        ganador(100)
    )

    resultados_eliminatoria[102] = resultado

    # Final

    final = simular_eliminatoria(
        ganador(101),
        ganador(102)
    )

    resultados_eliminatoria[104] = final

    return {

        "campeon":
            resultados_eliminatoria[104]["ganador"],

        "final":
            resultados_eliminatoria[104],

        "resultados":
            resultados_eliminatoria
    }

In [86]:
{
        "primeros": primeros,
        "segundos": segundos,
        "mejores_terceros": mejores_terceros,
        "equipos_por_grupo": equipos_por_grupo
    }

{'primeros':    posicion         team  PJ  PG  PE  PP  GF  GC  DG  PTS grupo
 0         1       Mexico   3   2   1   0   5   2   3    7     A
 0         1  Switzerland   3   2   1   0   5   3   2    7     B
 0         1      Morocco   3   2   0   1   8   7   1    6     C
 0         1       Turkey   3   2   1   0   7   5   2    7     D
 0         1      Ecuador   3   2   1   0  11   6   5    7     E
 0         1        Japan   3   3   0   0   6   2   4    9     F
 0         1         Iran   3   1   2   0   8   7   1    5     G
 0         1        Spain   3   3   0   0   8   3   5    9     H
 0         1       France   3   2   1   0   9   6   3    7     I
 0         1    Argentina   3   3   0   0  10   4   6    9     J
 0         1     Colombia   3   2   1   0   8   6   2    7     K
 0         1       Panama   3   2   1   0   5   3   2    7     L,
 'segundos':    posicion            team  PJ  PG  PE  PP  GF  GC  DG  PTS grupo
 1         2  Czech Republic   3   2   1   0   5   3   2    7 

In [87]:
resultado = simular_mundial()

In [88]:
resultado["campeon"]

'France'

In [89]:
resultado["final"]

{'local': 'Netherlands',
 'visitante': 'France',
 'goles_local': 0,
 'goles_visitante': 0,
 'ganador': 'France'}

In [90]:
for _ in range(5):

    print(
        simular_mundial()["campeon"]
    )

Argentina
Croatia
France
England
England


In [91]:
for _ in range(5):

    resultado = simular_mundial()

    print(
        resultado["final"]
    )

{'local': 'France', 'visitante': 'Brazil', 'goles_local': 2, 'goles_visitante': 1, 'ganador': 'France'}
{'local': 'Portugal', 'visitante': 'England', 'goles_local': 3, 'goles_visitante': 3, 'ganador': 'England'}
{'local': 'Norway', 'visitante': 'Croatia', 'goles_local': 2, 'goles_visitante': 1, 'ganador': 'Norway'}
{'local': 'France', 'visitante': 'Argentina', 'goles_local': 0, 'goles_visitante': 0, 'ganador': 'France'}
{'local': 'Spain', 'visitante': 'England', 'goles_local': 2, 'goles_visitante': 1, 'ganador': 'Spain'}


In [92]:
for _ in range(100):

    print(
        simular_mundial()["campeon"]
    )

Spain
Netherlands
Paraguay
Argentina
Japan
Colombia
Spain
Argentina
Spain
Brazil
Argentina
England
Spain
Colombia
England
Argentina
Argentina
England
Argentina
Argentina
France
Netherlands
Austria
Germany
Spain
Argentina
Spain
Argentina
Netherlands
Spain
Colombia
Argentina
Argentina
Argentina
France
England
Argentina
Spain
England
Argentina
Spain
Argentina
Netherlands
Spain
Spain
Argentina
Argentina
England
Norway
Argentina
France
Argentina
Brazil
Argentina
England
France
France
Argentina
Argentina
Switzerland
Spain
Argentina
Argentina
England
Argentina
Argentina
Argentina
Norway
Portugal
France
France
England
England
Ecuador
Spain
Netherlands
Colombia
Spain
Netherlands
Argentina
Germany
Portugal
Spain
England
Germany
Belgium
Spain
Spain
Ecuador
England
Croatia
Netherlands
Colombia
Argentina
Ecuador
Spain
Argentina
France
Spain
Argentina


In [93]:
from collections import Counter

campeones = []

for _ in range(100):

    campeones.append(
        simular_mundial()["campeon"]
    )

Counter(campeones).most_common(20)

[('Spain', 26),
 ('Argentina', 25),
 ('Colombia', 10),
 ('France', 10),
 ('Netherlands', 9),
 ('England', 6),
 ('Portugal', 4),
 ('Brazil', 3),
 ('Croatia', 2),
 ('Norway', 1),
 ('Morocco', 1),
 ('Ecuador', 1),
 ('Switzerland', 1),
 ('Iran', 1)]

In [94]:
from collections import Counter

campeones = []

for _ in range(1000):

    campeones.append(
        simular_mundial()["campeon"]
    )

ranking = Counter(
    campeones
).most_common(20)

ranking

KeyboardInterrupt: 

In [ ]:
resultado = simular_mundial()

len(
    resultado["resultados"]
)

31

In [ ]:
from collections import Counter

campeones = []

for _ in range(1000):

    campeones.append(
        simular_mundial()["campeon"]
    )

ranking = Counter(
    campeones
).most_common()

ranking

[('Spain', 251),
 ('Argentina', 202),
 ('England', 154),
 ('Colombia', 72),
 ('France', 69),
 ('Netherlands', 62),
 ('Ecuador', 34),
 ('Brazil', 30),
 ('Portugal', 23),
 ('Croatia', 18),
 ('Switzerland', 15),
 ('Germany', 9),
 ('Turkey', 9),
 ('Uruguay', 9),
 ('Norway', 8),
 ('Canada', 6),
 ('Morocco', 6),
 ('Japan', 6),
 ('Austria', 4),
 ('Belgium', 4),
 ('Senegal', 3),
 ('Paraguay', 3),
 ('Mexico', 1),
 ('Panama', 1),
 ('Iran', 1)]

In [ ]:
from collections import Counter

campeones = []

for _ in range(5000):

    campeones.append(
        simular_mundial()["campeon"]
    )

Counter(
    campeones
).most_common(20)

In [ ]:
selecciones.sort_values(
    "elo",
    ascending=False
)[
    ["team", "elo"]
].head(20)

,team,elo
28,Spain,2171.0
36,Argentina,2113.0
32,France,2062.0
44,England,2042.0
43,Colombia,1998.0
8,Brazil,1979.0
40,Portugal,1976.0
20,Netherlands,1959.0
19,Ecuador,1933.0
45,Croatia,1933.0


# Probabilidad de alcanzar cada ronda

In [95]:
def simular_mundial():

    (
        clasificaciones,
        partidos_grupos,
        primeros,
        segundos,
        terceros,
        cuartos
    ) = simular_fase_grupos(
        grupos
    )

    ranking_terceros = (
        terceros
        .sort_values(
            by=["PTS", "DG", "GF"],
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    mejores_terceros = (
        ranking_terceros.head(8)
    )

    equipos_por_grupo = {}

    for _, fila in primeros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "1º")
        ] = fila["team"]

    for _, fila in segundos.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "2º")
        ] = fila["team"]

    for _, fila in mejores_terceros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "3º")
        ] = fila["team"]

    terceros_ordenados = (
        mejores_terceros["team"]
        .tolist()
    )

    cruces_terceros = {

        74: (
            equipos_por_grupo[("E", "1º")],
            terceros_ordenados[0]
        ),

        77: (
            equipos_por_grupo[("I", "1º")],
            terceros_ordenados[1]
        ),

        79: (
            equipos_por_grupo[("A", "1º")],
            terceros_ordenados[2]
        ),

        80: (
            equipos_por_grupo[("L", "1º")],
            terceros_ordenados[3]
        ),

        81: (
            equipos_por_grupo[("D", "1º")],
            terceros_ordenados[4]
        ),

        82: (
            equipos_por_grupo[("G", "1º")],
            terceros_ordenados[5]
        ),

        85: (
            equipos_por_grupo[("B", "1º")],
            terceros_ordenados[6]
        ),

        87: (
            equipos_por_grupo[("K", "1º")],
            terceros_ordenados[7]
        )
    }

    cruces_fijos = {

        73: (
            equipos_por_grupo[("A", "2º")],
            equipos_por_grupo[("B", "2º")]
        ),

        75: (
            equipos_por_grupo[("F", "1º")],
            equipos_por_grupo[("C", "2º")]
        ),

        76: (
            equipos_por_grupo[("C", "1º")],
            equipos_por_grupo[("F", "2º")]
        ),

        78: (
            equipos_por_grupo[("E", "2º")],
            equipos_por_grupo[("I", "2º")]
        ),

        83: (
            equipos_por_grupo[("K", "2º")],
            equipos_por_grupo[("L", "2º")]
        ),

        84: (
            equipos_por_grupo[("H", "1º")],
            equipos_por_grupo[("J", "2º")]
        ),

        86: (
            equipos_por_grupo[("J", "1º")],
            equipos_por_grupo[("H", "2º")]
        ),

        88: (
            equipos_por_grupo[("D", "2º")],
            equipos_por_grupo[("G", "2º")]
        )
    }

    partidos_eliminatoria = {}

    for numero, (local, visitante) in cruces_fijos.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    for numero, (local, visitante) in cruces_terceros.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    resultados_eliminatoria = {}

    def ganador(numero_partido):

        return resultados_eliminatoria[
            numero_partido
        ]["ganador"]

    def jugar(numero_partido):

        local = partidos_eliminatoria[
            numero_partido
        ]["local"]

        visitante = partidos_eliminatoria[
            numero_partido
        ]["visitante"]

        resultado = simular_eliminatoria(
            local,
            visitante
        )

        resultados_eliminatoria[
            numero_partido
        ] = resultado

    # Dieciseisavos

    for partido in range(73, 89):

        jugar(partido)

    # Octavos

    cruces_octavos = {

        89: (74, 77),
        90: (73, 75),
        91: (76, 78),
        92: (79, 80),

        93: (83, 84),
        94: (81, 82),
        95: (86, 88),
        96: (85, 87)
    }

    for partido, (p1, p2) in cruces_octavos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Cuartos

    cruces_cuartos = {

        97: (89, 90),

        98: (93, 94),

        99: (91, 92),

        100: (95, 96)
    }

    for partido, (p1, p2) in cruces_cuartos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Semifinales

    resultado = simular_eliminatoria(
        ganador(97),
        ganador(98)
    )

    resultados_eliminatoria[101] = resultado

    resultado = simular_eliminatoria(
        ganador(99),
        ganador(100)
    )

    resultados_eliminatoria[102] = resultado

# Final

    final = simular_eliminatoria(
        ganador(101),
        ganador(102)
)

    resultados_eliminatoria[104] = final


# Equipos que alcanzan cada ronda

    equipos_16avos = set()

    for partido in range(73, 89):

        equipos_16avos.add(
        partidos_eliminatoria[partido]["local"]
    )

        equipos_16avos.add(
        partidos_eliminatoria[partido]["visitante"]
    )


    equipos_octavos = set()

    for partido in range(89, 97):

        equipos_octavos.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_octavos.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_cuartos = set()

    for partido in range(97, 101):

        equipos_cuartos.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_cuartos.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_semis = set()

    for partido in [101, 102]:

        equipos_semis.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_semis.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_final = {

        resultados_eliminatoria[104]["local"],
        resultados_eliminatoria[104]["visitante"]

}
    return {

    "campeon":
        resultados_eliminatoria[104]["ganador"],

    "equipos_16avos":
        equipos_16avos,

    "equipos_octavos":
        equipos_octavos,

    "equipos_cuartos":
        equipos_cuartos,

    "equipos_semis":
        equipos_semis,

    "equipos_final":
        equipos_final,

    "final":
        resultados_eliminatoria[104],

    "resultados":
        resultados_eliminatoria
}

In [96]:
from collections import defaultdict

contador_16avos = defaultdict(int)

contador_octavos = defaultdict(int)

contador_cuartos = defaultdict(int)

contador_semis = defaultdict(int)

contador_final = defaultdict(int)

contador_campeon = defaultdict(int)


n_simulaciones = 10


for _ in range(n_simulaciones):

    resultado = simular_mundial()

    for equipo in resultado["equipos_16avos"]:

        contador_16avos[equipo] += 1

    for equipo in resultado["equipos_octavos"]:

        contador_octavos[equipo] += 1

    for equipo in resultado["equipos_cuartos"]:

        contador_cuartos[equipo] += 1

    for equipo in resultado["equipos_semis"]:

        contador_semis[equipo] += 1

    for equipo in resultado["equipos_final"]:

        contador_final[equipo] += 1

    contador_campeon[
        resultado["campeon"]
    ] += 1

In [97]:
equipos = sorted(
    selecciones["team"].unique()
)

probabilidades = []

for equipo in equipos:

    probabilidades.append({

        "team": equipo,

        "16avos":
            contador_16avos[equipo]
            / n_simulaciones * 100,

        "Octavos":
            contador_octavos[equipo]
            / n_simulaciones * 100,

        "Cuartos":
            contador_cuartos[equipo]
            / n_simulaciones * 100,

        "Semifinal":
            contador_semis[equipo]
            / n_simulaciones * 100,

        "Final":
            contador_final[equipo]
            / n_simulaciones * 100,

        "Campeon":
            contador_campeon[equipo]
            / n_simulaciones * 100
    })

probabilidades = pd.DataFrame(
    probabilidades
)

probabilidades = probabilidades.sort_values(
    "Campeon",
    ascending=False
)

probabilidades.head(20)

,team,16avos,Octavos,Cuartos,Semifinal,Final,Campeon
40,Spain,100.0,100.0,80.0,70.0,60.0,40.0
16,England,100.0,90.0,80.0,60.0,30.0,30.0
1,Argentina,100.0,60.0,60.0,50.0,30.0,20.0
9,Colombia,100.0,80.0,40.0,30.0,20.0,10.0
4,Belgium,100.0,60.0,30.0,0.0,0.0,0.0
5,Bosnia and Herzegovina,70.0,0.0,0.0,0.0,0.0,0.0
2,Australia,70.0,20.0,0.0,0.0,0.0,0.0
0,Algeria,30.0,0.0,0.0,0.0,0.0,0.0
7,Canada,100.0,60.0,30.0,0.0,0.0,0.0
8,Cape Verde,20.0,0.0,0.0,0.0,0.0,0.0


In [98]:
resultado = simular_mundial()

print(resultado["campeon"])

Argentina


In [102]:
from collections import defaultdict

contador_16avos = defaultdict(int)

contador_octavos = defaultdict(int)

contador_cuartos = defaultdict(int)

contador_semis = defaultdict(int)

contador_final = defaultdict(int)

contador_campeon = defaultdict(int)


n_simulaciones = 1000


for _ in range(n_simulaciones):

    resultado = simular_mundial()

    for equipo in resultado["equipos_16avos"]:

        contador_16avos[equipo] += 1

    for equipo in resultado["equipos_octavos"]:

        contador_octavos[equipo] += 1

    for equipo in resultado["equipos_cuartos"]:

        contador_cuartos[equipo] += 1

    for equipo in resultado["equipos_semis"]:

        contador_semis[equipo] += 1

    for equipo in resultado["equipos_final"]:

        contador_final[equipo] += 1

    contador_campeon[
        resultado["campeon"]
    ] += 1

In [103]:
equipos = sorted(
    selecciones["team"].unique()
)

probabilidades = []

for equipo in equipos:

    probabilidades.append({

        "team": equipo,

        "16avos":
            contador_16avos[equipo]
            / n_simulaciones * 100,

        "Octavos":
            contador_octavos[equipo]
            / n_simulaciones * 100,

        "Cuartos":
            contador_cuartos[equipo]
            / n_simulaciones * 100,

        "Semifinal":
            contador_semis[equipo]
            / n_simulaciones * 100,

        "Final":
            contador_final[equipo]
            / n_simulaciones * 100,

        "Campeon":
            contador_campeon[equipo]
            / n_simulaciones * 100
    })

probabilidades = pd.DataFrame(
    probabilidades
)

probabilidades = probabilidades.sort_values(
    "Campeon",
    ascending=False
)

probabilidades.head(20)

,team,16avos,Octavos,Cuartos,Semifinal,Final,Campeon
1,Argentina,100.0,80.3,70.7,57.0,39.1,26.9
40,Spain,100.0,85.2,62.2,54.4,40.5,24.4
16,England,99.9,83.6,65.7,46.2,27.3,14.9
17,France,99.3,78.2,49.9,32.3,15.6,7.2
9,Colombia,99.4,74.7,42.5,19.2,10.1,4.7
28,Netherlands,99.8,67.3,47.8,27.9,12.3,3.2
6,Brazil,99.8,62.3,37.3,16.3,6.1,3.2
10,Croatia,98.4,52.6,23.5,14.3,6.3,2.7
33,Portugal,99.1,64.5,34.2,15.4,7.5,2.6
14,Ecuador,99.0,69.5,38.6,20.1,7.4,2.5


In [ ]:
resultado = simular_mundial()

print(resultado["campeon"])

In [98]:
from collections import defaultdict

contador_16avos = defaultdict(int)

contador_octavos = defaultdict(int)

contador_cuartos = defaultdict(int)

contador_semis = defaultdict(int)

contador_final = defaultdict(int)

contador_campeon = defaultdict(int)


n_simulaciones = 500


for _ in range(n_simulaciones):

    resultado = simular_mundial()

    for equipo in resultado["equipos_16avos"]:

        contador_16avos[equipo] += 1

    for equipo in resultado["equipos_octavos"]:

        contador_octavos[equipo] += 1

    for equipo in resultado["equipos_cuartos"]:

        contador_cuartos[equipo] += 1

    for equipo in resultado["equipos_semis"]:

        contador_semis[equipo] += 1

    for equipo in resultado["equipos_final"]:

        contador_final[equipo] += 1

    contador_campeon[
        resultado["campeon"]
    ] += 1

In [104]:
probabilidades_export = (
    probabilidades
    .round(1)
)

probabilidades_export.to_csv(
    "probabilidades_mundial_500_2.csv",
    index=False
)
probabilidades.to_excel(
    "probabilidades_mundial_500_2.xlsx",
    index=False
)

In [99]:
equipos = sorted(
    selecciones["team"].unique()
)

probabilidades = []

for equipo in equipos:

    probabilidades.append({

        "team": equipo,

        "16avos":
            contador_16avos[equipo]
            / n_simulaciones * 100,

        "Octavos":
            contador_octavos[equipo]
            / n_simulaciones * 100,

        "Cuartos":
            contador_cuartos[equipo]
            / n_simulaciones * 100,

        "Semifinal":
            contador_semis[equipo]
            / n_simulaciones * 100,

        "Final":
            contador_final[equipo]
            / n_simulaciones * 100,

        "Campeon":
            contador_campeon[equipo]
            / n_simulaciones * 100
    })

probabilidades = pd.DataFrame(
    probabilidades
)

probabilidades = probabilidades.sort_values(
    "Campeon",
    ascending=False
)

probabilidades.head(20)

,team,16avos,Octavos,Cuartos,Semifinal,Final,Campeon
40,Spain,100.0,86.0,63.0,54.0,39.2,26.4
1,Argentina,100.0,79.8,68.8,51.8,34.8,20.8
16,England,100.0,82.2,66.4,46.4,28.0,14.8
9,Colombia,99.8,75.8,44.0,21.8,11.4,6.2
28,Netherlands,99.6,71.8,53.4,26.0,12.6,5.6
17,France,99.2,73.8,47.2,32.4,14.2,5.0
6,Brazil,99.4,63.4,40.0,21.2,9.2,4.6
14,Ecuador,99.8,68.0,34.2,18.2,8.2,3.6
10,Croatia,99.0,56.4,27.8,13.2,6.4,2.4
33,Portugal,98.0,62.0,27.6,13.2,6.0,2.4


# Resultados Monte Carlo

Se realizaron 1000 simulaciones completas del Mundial 2026.

Las probabilidades obtenidas representan la frecuencia con la que cada selección alcanzó cada ronda del torneo.

Las selecciones con mayor probabilidad de proclamarse campeonas fueron:

1. España
2. Argentina
3. Inglaterra
4. Francia
5. Países Bajos

IDEA PARA LA VISUALIZACION

In [ ]:
from collections import Counter

campeones = []

for _ in range(10000):

    campeones.append(
        simular_mundial()["campeon"]
    )

ranking = Counter(
    campeones
).most_common()

In [ ]:
ranking_df = pd.DataFrame(
    ranking,
    columns=[
        "team",
        "titulos"
    ]
)

ranking_df.to_csv(
    "montecarlo_10000.csv",
    index=False
)